# 🔹 Balanços de Conservação e Números Adimensionais
## Volume I – Capítulo 3 | Nível: Graduação/Pós-Graduação

**Objetivos de aprendizagem:**
- Aplicar a analogia da conta bancária para balanços de transporte
- Diferenciar propriedades extensivas e intensivas
- Calcular números de Reynolds, Péclet, Prandtl e Schmidt
- Classificar regimes de escoamento e transporte

**Referência:** Lugon Junior (2026), *Fundamentos de Fenômenos de Transporte*, Vol I, Cap. 3.

In [ ]:
# Importações padrão
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants

# Configurações de estilo
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 🔸 Analogia da Conta Bancária

O balanço de qualquer propriedade $B$ em um volume de controle segue:

$$\frac{dB}{dt} = \underbrace{\dot{B}_{ent}}_{\text{entradas}} - \underbrace{\dot{B}_{sai}}_{\text{saídas}} + \underbrace{\dot{G}}_{\text{geração}} - \underbrace{\dot{C}}_{\text{consumo}}$$

### Exemplo Interativo: Tanque com entrada e saída

In [ ]:
def balanco_tanque(Q_ent, Q_sai, A_base, h_inicial, h_final, dt=1.0):
    """
    Simula a evolução do nível em um tanque com balanço de volume.
    
    Parâmetros:
    -----------
    Q_ent, Q_sai : float
        Vazões de entrada e saída [m³/s]
    A_base : float
        Área da base do tanque [m²]
    h_inicial, h_final : float
        Níveis inicial e final [m]
    dt : float, optional
        Passo de tempo [s]
    
    Retorna:
    --------
    t, h : arrays
        Tempo e nível para plotagem
    """
    dVdt = Q_ent - Q_sai  # Variação de volume [m³/s]
    dhdt = dVdt / A_base   # Variação de nível [m/s]
    
    if dhdt == 0:
        return np.array([0]), np.array([h_inicial])
    
    # Tempo necessário para atingir h_final
    t_total = abs(h_final - h_inicial) / abs(dhdt)
    n_steps = int(t_total / dt) + 1
    
    t = np.linspace(0, t_total, n_steps)
    h = h_inicial + dhdt * t
    
    return t, h

# Exercício Vol I, Cap 3, Ex 2
Q_ent = 0.8 / 1000    # 0.8 L/s → m³/s
Q_sai = 0.5 / 1000    # 0.5 L/s → m³/s
A = 4.0               # m²
h0, hf = 0.5, 1.2     # m

t, h = balanco_tanque(Q_ent, Q_sai, A, h0, hf)

plt.figure(figsize=(8, 5))
plt.plot(t/60, h, 'b-', linewidth=2)  # tempo em minutos
plt.xlabel('Tempo [min]')
plt.ylabel('Nível h [m]')
plt.title('Evolução do nível no tanque')
plt.grid(True, alpha=0.3)
plt.axhline(y=hf, color='r', linestyle='--', label=f'h_final = {hf} m')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Tempo para atingir {hf} m: {t[-1]/60:.1f} min ≈ {t[-1]/3600:.2f} h")

## 🔸 Números Adimensionais Fundamentais

### Função de cálculo genérico

In [ ]:
def calcular_numeros_adimensionais(U, L, nu, D=None, alpha=None, kappa=None, g=9.81):
    """
    Calcula números adimensionais fundamentais.
    
    Parâmetros:
    -----------
    U : float
        Velocidade característica [m/s]
    L : float
        Comprimento característico [m]
    nu : float
        Viscosidade cinemática [m²/s]
    D : float, optional
        Difusividade mássica [m²/s]
    alpha : float, optional
        Difusividade térmica [m²/s]
    kappa : float, optional
        Condutividade térmica [W/(m·K)]
    g : float, optional
        Aceleração da gravidade [m/s²]
    
    Retorna:
    --------
    dict : Dicionário com números adimensionais calculados
    """
    resultados = {}
    
    # Reynolds: inércia/viscosidade
    resultados['Re'] = U * L / nu
    
    # Péclet (massa ou calor)
    if D is not None:
        resultados['Pe_massa'] = U * L / D
    if alpha is not None:
        resultados['Pe_calor'] = U * L / alpha
    
    # Prandtl: nu/alpha
    if alpha is not None and nu is not None:
        resultados['Pr'] = nu / alpha
    
    # Schmidt: nu/D
    if D is not None and nu is not None:
        resultados['Sc'] = nu / D
    
    # Froude: inércia/gravidade
    resultados['Fr'] = U / np.sqrt(g * L)
    
    return resultados

# Exercício Vol I, Cap 3, Ex 3: Rio
U = 0.6      # m/s
L = 15.0     # m (largura característica)
nu = 1e-6    # m²/s (água a 20°C)
E_L = 8.0    # m²/s (dispersão longitudinal)

nums = calcular_numeros_adimensionais(U, L, nu, D=E_L)

print(f"Número de Reynolds: Re = {nums['Re']:.2e}")
print(f"Número de Péclet: Pe = {nums['Pe_massa']:.2f}")
print(f"Número de Froude: Fr = {nums['Fr']:.3f}")
print(f"\nClassificação:")
print(f"  - Re > 2400 → Escoamento TURBULENTO ✓")
print(f"  - Pe > 1 → Transporte ADVETIVO dominante ✓")

## 🔸 Diagrama de Regimes Re × Pe

Visualização da classificação combinada de regime hidrodinâmico e mecanismo de transporte.

In [ ]:
# Criar grade para o diagrama
Re_vals = np.logspace(1, 7, 100)
Pe_vals = np.logspace(-2, 4, 100)
Re_grid, Pe_grid = np.meshgrid(Re_vals, Pe_vals)

# Classificação por região
regimes = np.empty_like(Re_grid, dtype=object)
for i in range(Re_grid.shape[0]):
    for j in range(Re_grid.shape[1]):
        if Re_grid[i,j] < 2000:
            hid = 'Laminar'
        elif Re_grid[i,j] < 2400:
            hid = 'Transição'
        else:
            hid = 'Turbulento'
        
        transp = 'Advectivo' if Pe_grid[i,j] > 1 else 'Difusivo'
        regimes[i,j] = f"{hid}\n{transp}"

plt.figure(figsize=(10, 8))

# Cores para os regimes
colors = {'Laminar\nDifusivo': '#a6cee3', 'Laminar\nAdvectivo': '#1f78b4',
          'Transição\nDifusivo': '#b2df8a', 'Transição\nAdvectivo': '#33a02c',
          'Turbulento\nDifusivo': '#fb9a99', 'Turbulento\nAdvectivo': '#e31a1c'}

# Plotar regiões
for label, color in colors.items():
    mask = np.array([[regimes[i,j] == label for j in range(regimes.shape[1])] 
                     for i in range(regimes.shape[0])])
    if np.any(mask):
        plt.contourf(Re_grid, Pe_grid, mask.astype(int), levels=[0.5, 1.5], 
                    colors=[color], alpha=0.3)

# Linhas de referência
plt.axvline(x=2000, color='gray', linestyle='--', alpha=0.5, label='Re = 2000')
plt.axvline(x=2400, color='gray', linestyle=':', alpha=0.5, label='Re = 2400')
plt.axhline(y=1, color='black', linestyle='-', linewidth=1.5, label='Pe = 1')

# Casos de estudo
casos = {
    'Microcanal': {'Re': 100, 'Pe': 0.5, 'marker': 'o'},
    'Tubulação': {'Re': 5e4, 'Pe': 1e3, 'marker': 's'},
    'Rio Macaé': {'Re': 1.4e5, 'Pe': 1.125, 'marker': '^'}
}

for nome, dados in casos.items():
    plt.scatter(dados['Re'], dados['Pe'], marker=dados['marker'], 
               s=100, edgecolors='black', label=nome)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Número de Reynolds (Re)', fontsize=12)
plt.ylabel('Número de Péclet (Pe)', fontsize=12)
plt.title('Diagrama de Regimes: Hidrodinâmica × Transporte', fontsize=14)
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## ✅ Exercícios Propostos

1. **[Graduação]** Classifique as grandezas: pressão, volume total, temperatura, quantidade de movimento, concentração molar.
2. **[Pós-Graduação]** Adimensionalize a equação de advecção-difusão 1D e mostre que Pe é o único parâmetro resultante.

> 💡 Dica: Use a função `calcular_numeros_adimensionais()` como base para os cálculos.